# euclid_cutouts — Tutorial

This notebook shows how to use the `euclid_cutouts` library to produce colour
cutout images from Euclid data. Three workflows are demonstrated:

1. **Single cutout** — render one (4, H, W) IYJH array through azulero / bulk_euclid
2. **FITS-input mode** — batch-render a directory of pre-made FITS cutout files
3. **ML-inference loop** — load FITS cutouts on the fly, run a classifier, and
   keep only the high-confidence detections (FITS + colour images)

## Install

```bash
pip install -e /media/user/euclid_cutouts
```

In [ ]:
import os
import numpy as np
from astropy.io import fits
from PIL import Image

from euclid_cutouts import (
    load_fits_cutout,
    render_azulero,
    render_bulk_variant,
    render_cutout,
    render_fits_dir,
)

# Paths to the local clones — needed so the renderers can find azulero_render
# and the bulk_euclid utilities. Adjust if yours are elsewhere.
CUTANA_ROOT = "/media/user/astronomaly-euclid"
BULK_ROOT   = "/media/user/bulk-euclid-cutouts"

## 1. Render a single cutout

`render_cutout` takes a `(4, H, W)` IYJH float32 array (band order: VIS,
NIR-Y, NIR-J, NIR-H) and returns a dict of `{name: (H, W, 3) uint8 RGB}`
arrays — one per enabled renderer/variant.

You can feed it data from any source: a Cutana FITS file, a tile extraction,
or a numpy array you built yourself.

In [ ]:
# --- Example with a real Cutana FITS cutout (single-band VIS) ---
# If you have a multi-band FITS cutout, load it directly:
#
#   iyjh = load_fits_cutout("path/to/cutout.fits",
#                           band_order=["VIS", "NIR_Y", "NIR_J", "NIR_H"])

# For the demo we synthesise a fake 4-band cutout:
rng = np.random.default_rng(42)
iyjh_fake = (rng.random((4, 64, 64), dtype=np.float32) * 50).clip(0.1)

# Render through all available stretches at once
results = render_cutout(
    iyjh_fake,
    renderers=["azulero", "bulk_euclid"],
    bulk_variants=["gz_arcsinh_vis_y", "sw_mtf_vis_y_j"],
    cutana_root=CUTANA_ROOT,
    bulk_euclid_root=BULK_ROOT,
)

print("Renderers produced:", list(results.keys()))
for name, rgb in results.items():
    print(f"  {name}: shape={rgb.shape}  dtype={rgb.dtype}")

# Display side by side
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4))
if len(results) == 1:
    axes = [axes]
for ax, (name, rgb) in zip(axes, results.items()):
    ax.imshow(rgb, origin="upper")
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

You can also call the individual renderers directly if you only need one:

```python
rgb_az = render_azulero(iyjh, cutana_root=CUTANA_ROOT)
rgb_gz = render_bulk_variant("gz_arcsinh_vis_y", vis, y, j, bulk_euclid_root=BULK_ROOT)
```

## 2. Batch-render a directory of FITS cutouts

If you have a folder of pre-made FITS cutout files (e.g. Cutana output), use
`render_fits_dir` to colour-render all of them in one call.  It handles
multiprocessing, skip-existing, and progress bars.

In [ ]:
# --- Create a small demo directory with fake FITS cutouts ---
import tempfile, shutil

demo_fits_dir = os.path.join(tempfile.gettempdir(), "euclid_cutouts_demo_fits")
demo_out_dir  = os.path.join(tempfile.gettempdir(), "euclid_cutouts_demo_out")
shutil.rmtree(demo_fits_dir, ignore_errors=True)
shutil.rmtree(demo_out_dir, ignore_errors=True)
os.makedirs(demo_fits_dir)

rng = np.random.default_rng(0)
for i in range(5):
    primary = fits.PrimaryHDU()
    hdus = [primary]
    for band_idx in range(4):
        data = (rng.random((64, 64), dtype=np.float32) * 80).clip(0.1)
        hdus.append(fits.ImageHDU(data=data, name=f"CHANNEL_{band_idx+1}"))
    fits.HDUList(hdus).writeto(os.path.join(demo_fits_dir, f"source_{i:04d}.fits"),
                               overwrite=True)

print(f"Created {len(os.listdir(demo_fits_dir))} demo FITS files in {demo_fits_dir}")

In [ ]:
# --- Batch render ---
totals = render_fits_dir(
    input_dir=demo_fits_dir,
    output_dir=demo_out_dir,
    enable_azulero=True,
    enable_bulk_euclid=True,
    bulk_variants=["gz_arcsinh_vis_y", "sw_mtf_vis_y_j"],
    band_order=["VIS", "NIR_Y", "NIR_J", "NIR_H"],
    fmt="jpg",
    jpeg_quality=95,
    n_workers=1,
    progress_bar=True,
    cutana_root=CUTANA_ROOT,
    bulk_euclid_root=BULK_ROOT,
)

print("\nTotals:", totals)
print("\nOutput tree:")
for root, dirs, files in os.walk(demo_out_dir):
    level = root.replace(demo_out_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files)[:3]:
        print(f"{indent}  {f}")
    if len(files) > 3:
        print(f"{indent}  ... ({len(files)} files total)")

### With Cutana 3-band output

If your FITS cutouts come from Cutana with the standard workspace band order
(NIR-Y, NIR-J, VIS — no NIR-H), just set `band_order` accordingly.
Missing bands are zero-filled automatically.

```python
totals = render_fits_dir(
    input_dir="/path/to/cutana_output",
    output_dir="cutouts",
    band_order=["NIR_Y", "NIR_J", "VIS"],  # Cutana 3-band order
    enable_azulero=True,
    enable_bulk_euclid=True,
    bulk_variants=["gz_arcsinh_vis_y"],
    cutana_root=CUTANA_ROOT,
    bulk_euclid_root=BULK_ROOT,
)
```

## 3. CSV-mode (tile-based pipeline via CLI)

The original tile-based pipeline reads a CSV of sources, resolves tiles via
HEALPix, loads full tile FITS, extracts cutouts, and renders through all four
colour stretches (azulero, eummy, bulk_euclid). This mode also supports eummy,
which needs tile-level FITS on disk.

Run from the terminal:

```bash
# Edit the CONFIG block at the top of the script first, then:
python scripts/make_colour_cutouts.py
```

Or switch to FITS-input mode from the same script by setting
`INPUT_FITS_DIR` in the CONFIG block — the script then calls `render_fits_dir`
internally.

## 4. ML-inference loop — render on the fly, keep high-confidence detections

This is the key use case for the library API: you have a trained classifier
(e.g. a strong-lens finder) and a directory of FITS cutouts.  You want to:

1. Load each FITS cutout
2. Run your model on the raw array (or on a rendered RGB)
3. **Only if** the predicted probability exceeds a threshold, save the
   original FITS **and** the colour images

This avoids rendering and saving thousands of uninteresting cutouts.

In [ ]:
def ml_filter_and_render(
    fits_dir: str,
    output_dir: str,
    predict_fn,
    *,
    threshold: float = 0.5,
    band_order: list[str] | None = None,
    renderers: list[str] | None = None,
    bulk_variants: list[str] | None = None,
    fmt: str = "jpg",
    jpeg_quality: int = 95,
    save_fits: bool = True,
    cutana_root: str | None = None,
    bulk_euclid_root: str | None = None,
):
    """Load FITS cutouts, run an ML model, and save only high-confidence hits.

    Parameters
    ----------
    fits_dir : directory of .fits cutout files.
    output_dir : root output dir. Colour images go into subdirs per renderer;
        FITS copies go into ``output_dir/fits/``.
    predict_fn : callable(iyjh) -> float.  Receives a (4, H, W) float32 array,
        returns a scalar probability (0–1).  This is your model.
    threshold : minimum probability to keep a cutout.
    band_order : FITS extension-to-band mapping (default 4-band IYJH).
    renderers : list of renderers ("azulero", "bulk_euclid").
    bulk_variants : which bulk_euclid variants to produce.
    fmt : output image format ("jpg" or "png").
    jpeg_quality : JPEG quality.
    save_fits : if True, copy the original FITS file into output_dir/fits/.
    cutana_root : path to astronomaly-euclid.
    bulk_euclid_root : path to bulk-euclid-cutouts.

    Returns
    -------
    results : list of dicts with keys: stem, fits_path, probability, kept.
    """
    import glob, shutil

    if renderers is None:
        renderers = ["azulero", "bulk_euclid"]
    if bulk_variants is None:
        bulk_variants = ["gz_arcsinh_vis_y", "sw_mtf_vis_y_j"]
    if band_order is None:
        band_order = ["VIS", "NIR_Y", "NIR_J", "NIR_H"]

    fits_files = sorted(glob.glob(os.path.join(fits_dir, "*.fits")))
    print(f"Scanning {len(fits_files)} FITS files, threshold={threshold}")

    # Prepare output dirs
    fits_out = os.path.join(output_dir, "fits")
    if save_fits:
        os.makedirs(fits_out, exist_ok=True)
    colour_dirs = {}
    if "azulero" in renderers:
        d = os.path.join(output_dir, "azulero")
        os.makedirs(d, exist_ok=True)
        colour_dirs["azulero"] = d
    if "bulk_euclid" in renderers:
        for v in bulk_variants:
            d = os.path.join(output_dir, v)
            os.makedirs(d, exist_ok=True)
            colour_dirs[v] = d

    save_kw = {"quality": jpeg_quality} if fmt == "jpg" else {}
    results = []
    n_kept = 0

    for fits_path in fits_files:
        stem = os.path.splitext(os.path.basename(fits_path))[0]

        # 1. Load
        iyjh = load_fits_cutout(fits_path, band_order=band_order)
        if iyjh is None:
            results.append({"stem": stem, "fits_path": fits_path,
                            "probability": 0.0, "kept": False})
            continue

        # 2. Predict
        prob = float(predict_fn(iyjh))

        # 3. Filter
        kept = prob >= threshold
        results.append({"stem": stem, "fits_path": fits_path,
                        "probability": prob, "kept": kept})
        if not kept:
            continue

        # 4. Save FITS
        if save_fits:
            shutil.copy2(fits_path, os.path.join(fits_out, f"{stem}.fits"))

        # 5. Render and save colour images
        colour_results = render_cutout(
            iyjh,
            renderers=renderers,
            bulk_variants=bulk_variants,
            cutana_root=cutana_root,
            bulk_euclid_root=bulk_euclid_root,
        )
        for name, rgb in colour_results.items():
            out_path = os.path.join(colour_dirs[name], f"{stem}.{fmt}")
            Image.fromarray(rgb).save(out_path, **save_kw)

        n_kept += 1

    print(f"Done: {n_kept}/{len(fits_files)} kept (threshold={threshold})")
    return results

In [ ]:
# --- Demo: fake model that returns random probabilities ---
# Replace this with your actual model (e.g. a PyTorch/TF classifier).
#
# Real example with a PyTorch model:
#
#   import torch
#   model = torch.load("lens_classifier.pt").eval()
#   def predict_fn(iyjh):
#       x = torch.from_numpy(iyjh).unsqueeze(0)   # (1, 4, H, W)
#       with torch.no_grad():
#           return model(x).sigmoid().item()

rng_model = np.random.default_rng(99)
def fake_predict(iyjh):
    """Dummy: returns a random probability based on the source index."""
    return rng_model.random()

ml_out_dir = os.path.join(tempfile.gettempdir(), "euclid_cutouts_demo_ml")
shutil.rmtree(ml_out_dir, ignore_errors=True)

results = ml_filter_and_render(
    fits_dir=demo_fits_dir,
    output_dir=ml_out_dir,
    predict_fn=fake_predict,
    threshold=0.6,
    renderers=["azulero", "bulk_euclid"],
    bulk_variants=["gz_arcsinh_vis_y"],
    cutana_root=CUTANA_ROOT,
    bulk_euclid_root=BULK_ROOT,
)

In [ ]:
# --- Inspect results ---
import pandas as pd

df_results = pd.DataFrame(results)
print(df_results[["stem", "probability", "kept"]].to_string(index=False))

print(f"\nKept cutouts saved to: {ml_out_dir}")
for sub in ["fits", "azulero", "gz_arcsinh_vis_y"]:
    d = os.path.join(ml_out_dir, sub)
    if os.path.isdir(d):
        print(f"  {sub}/: {len(os.listdir(d))} files")

In [ ]:
# --- Display the kept detections ---
kept = [r for r in results if r["kept"]]
if kept:
    fig, axes = plt.subplots(1, len(kept), figsize=(4 * len(kept), 4),
                             squeeze=False)
    for ax, r in zip(axes[0], kept):
        img_path = os.path.join(ml_out_dir, "azulero", f"{r['stem']}.jpg")
        ax.imshow(Image.open(img_path))
        ax.set_title(f"{r['stem']}\np={r['probability']:.3f}")
        ax.axis("off")
    plt.suptitle("Kept detections (azulero)")
    plt.tight_layout()
    plt.show()
else:
    print("No detections above threshold — try lowering it.")

## API summary

| Function | Input | Output | Use case |
|---|---|---|---|
| `load_fits_cutout(path, band_order)` | FITS file path | `(4, H, W)` float32 or None | Load any multi-extension FITS cutout |
| `render_azulero(iyjh)` | `(4, H, W)` float32 | `(H, W, 3)` uint8 RGB | Single azulero colour image |
| `render_bulk_variant(variant, vis, y, j)` | per-band 2D arrays | `(H, W, 3)` uint8 RGB | Single bulk_euclid variant |
| `render_cutout(iyjh, renderers, ...)` | `(4, H, W)` float32 | `{name: RGB}` dict | Multi-renderer in one call |
| `render_fits_dir(input_dir, output_dir, ...)` | directory of FITS | `{name: count}` dict | Batch rendering with multiprocessing |

In [ ]:
# --- Cleanup demo files ---
shutil.rmtree(demo_fits_dir, ignore_errors=True)
shutil.rmtree(demo_out_dir, ignore_errors=True)
shutil.rmtree(ml_out_dir, ignore_errors=True)
print("Cleaned up temp dirs.")